# A whole detector in fast mode, against the clock

Fits every tile of one LSSTCam detector with `FAST_STELLAR_FIELD_CONFIG` (one pass with the visit
PSF, detection 5 sigma -> 3 sigma, bright-star terms kept, the end-of-pass tests off) and
pipe_crowd's sparse solver, using every CPU this notebook server has.

1. setup and build check;
2. how many CPUs this server gives you;
3. one tile, to predict the whole-detector time before committing to it;
4. the whole detector, timed end to end (loading included), then stitched and plotted.

Run with the **LSST** kernel; restart the kernel after the build cell if it builds.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd_v2"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

## 1. CPUs and memory of this server

In [ ]:
try:
    q, p = open("/sys/fs/cgroup/cpu.max").read().split()
    cpuLimit = None if q == "max" else int(q) / int(p)
except (FileNotFoundError, ValueError):
    cpuLimit = None
visible = len(os.sched_getaffinity(0))
N_JOBS = max(1, int(cpuLimit) if cpuLimit else visible)
try:
    import psutil
    mem = f"{psutil.virtual_memory().total / 2**30:.0f} GB"
except ImportError:
    mem = "unknown"
print(f"CPU limit: {cpuLimit or 'none'} | CPUs visible: {visible} | memory: {mem} -> N_JOBS = {N_JOBS}")

## 2. Settings and the visit image

`OUT` stays the same between runs so the Gaia catalogue is read from its cache
(`OUT/gaia_cache/`) after the first run; tiles are refitted each time (`overwrite=True`).

In [ ]:
VISIT, DETECTOR = 2025062100433, 144
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
OUT = "fast_detector"
CFG = crowd.makeConfig(crowd.FAST_STELLAR_FIELD_CONFIG)

T_START = time.perf_counter()             # the clock for the whole run starts here
from lsst.daf.butler import Butler  # noqa: E402
butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])
tLoad = time.perf_counter() - T_START
bb = visitImage.getBBox()
nTiles = len(crowd.tileSpecs(bb.getWidth(), bb.getHeight(), 400, bb.getMinX(), bb.getMinY(), None))
print(f"visit image loaded in {tLoad:.1f} s; {nTiles} tiles")

## 3. One tile first: predict the whole-detector time

Fits the tile containing (1150, 1150) on one CPU, after an untimed warm-up run that loads and caches
the Gaia catalogue.  The detector is cut into 11 x 11 tiles of ~372 px; the whole detector takes
roughly load + ceil(tiles / N_JOBS) x (time per tile), more if some tiles are more crowded.

In [ ]:
import contextlib, io  # noqa: E401,E402
# warm-up (not timed): loads the Gaia catalogue once and caches it in OUT/gaia_cache/
with contextlib.redirect_stdout(io.StringIO()):
    crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                   region=(1150, 1151, 1150, 1151), overwrite=True, plots=None, repo=REPO,
                   collection=COLLECTION)
t0 = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    done = crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                          region=(1150, 1151, 1150, 1151), overwrite=True, plots=None, repo=REPO,
                          collection=COLLECTION)   # a 2 px region: exactly one tile
tTile = (time.perf_counter() - t0) / max(len(done), 1)
est = tLoad + np.ceil(nTiles / N_JOBS) * tTile
print(f"one tile: {tTile:.1f} s -> {nTiles} tiles with {N_JOBS} workers: about {est:.0f} s "
      f"({est / 60:.1f} min), if every tile is like this one")

## 4. The whole detector

All tiles in `N_JOBS` parallel worker processes; each writes its tile file and a log in
`OUT/logs/`.  The total printed at the end is loading the visit image (cell 2) + fitting +
stitching; the one-tile test of cell 3 is not counted.

In [ ]:
crowd.solverStats(reset=True, verbose=False)
t0 = time.perf_counter()
paths = crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                       nJobs=N_JOBS, overwrite=True, plots=None, repo=REPO, collection=COLLECTION)
tFit = time.perf_counter() - t0
t0 = time.perf_counter()
mosaic = crowd.stitchTiles(OUT, verbose=False)
tStitch = time.perf_counter() - t0
tTotal = tLoad + tFit + tStitch                         # cell 3's test run not counted
print(f"\n{len(paths)}/{nTiles} tiles | load {tLoad:.0f} s + fit {tFit:.0f} s + stitch {tStitch:.0f} s "
      f"= {tTotal:.0f} s ({tTotal / 60:.1f} min) | {len(mosaic['catalog'])} sources")

In [ ]:
crowd.plotDetector(mosaic)
plt.show()
crowd.plotDetector(mosaic, view=(1000, 1400, 1000, 1400), binning=1)
plt.show()

### Slowest tiles

Each tile's log ends with its time; the slowest set the total when there are more tiles than
workers.

In [ ]:
import glob, re  # noqa: E401,E402
times = []
for log in glob.glob(os.path.join(OUT, "logs", "*.log")):
    m = re.findall(r"wall clock: ([0-9.]+)s", open(log).read())
    if m:
        times.append((float(m[-1]), os.path.basename(log)[:-4]))
times.sort(reverse=True)
if times:
    print(f"per tile: median {np.median([t for t, _ in times]):.1f} s, max {times[0][0]:.1f} s")
    for t, name in times[:8]:
        print(f"  {name}: {t:.1f} s")